# Chapter 21 Companion Notebook: Embeddings and Similarity in Business Analytics

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch21_Embeddings_and_Similarity.ipynb)

This notebook accompanies Chapter 21 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




This notebook is designed as a classroom appendix. It uses synthetic business data so that every step can run safely in Google Colab without external files, paid APIs, or private customer information. The goal is not to build a production-scale vector database. The goal is to make the measurement logic of embeddings, similarity, retrieval, matching, and governance visible.

## Why this matters (business framing)

Embeddings help analysts compare messy business objects that do not naturally live in spreadsheets: customer comments, support tickets, product IDs, purchase baskets, and journey events. Once those objects become vectors, a wide range of tasks become similarity problems. A support manager can retrieve related complaints, a brand team can inspect nearby themes, a merchandising team can map product neighborhoods, and an operations team can deduplicate repeated tickets.

The risk is that similarity can look objective even when it reflects weak preprocessing, biased coverage, vague units of analysis, or poorly chosen thresholds. This notebook therefore treats embeddings as a measurement system. We will build small offline embedding workflows, evaluate them against decision-oriented anchors, compare keyword and dense retrieval, calibrate matching thresholds, and document governance checks.

## Agenda

1. Setup and reproducibility
2. Synthetic business text corpus with metadata
3. Similarity metrics: cosine, dot product, Euclidean distance, and Jaccard overlap
4. Sparse lexical vectors versus dense document embeddings
5. Word embedding intuition from context and contrast
6. Sentence and document retrieval with decision-oriented evaluation
7. Hybrid retrieval and metadata-aware refinement
8. Matching and threshold calibration
9. Item, customer, and event embeddings beyond text
10. Operational checkpoints: anchors, coverage, drift, privacy, and documentation
11. Exercises

## Connection map

Chapter 20 prepared the corpus and emphasized data quality. Chapter 21 turns the cleaned corpus into representations that support comparison. Later chapters will use these same ideas for topic modeling, sentiment classification, transformer-based NLP, and retrieval-augmented text mining. In practice, embedding design sits between preprocessing and downstream modeling. If the representation is weak, every retrieval, clustering, matching, or recommendation workflow built on top of it becomes fragile.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# - install missing packages if needed
# - import libraries
# - set seeds
# - configure output folders
# ============================================================

import importlib.util
import subprocess
import sys
from pathlib import Path

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}

for import_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import math
import os
import random
import re
import warnings
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("/content/ch21_outputs") if Path("/content").exists() else Path("ch21_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)

print("Setup complete")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## Utility functions

These helpers keep the main sections focused on representation and business decisions. They handle text normalization, tokenization, similarity ranking, retrieval evaluation, co-occurrence embeddings, and compact plotting.

In [ ]:
# ============================================================
# Utility functions for text processing, similarity, retrieval, and plotting
# ============================================================

STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "because", "but", "by", "can", "for", "from", "had", "has",
    "have", "how", "i", "in", "is", "it", "its", "me", "my", "of", "on", "or", "our", "please", "so", "that",
    "the", "their", "them", "there", "this", "to", "was", "we", "were", "what", "when", "where", "who", "why",
    "will", "with", "you", "your", "about", "after", "again", "still", "not"
}


def print_section(title):
    print("=" * len(title))
    print(title)
    print("=" * len(title))


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+", " url ", text)
    text = re.sub(r"[\w\.-]+@[\w\.-]+", " email ", text)
    text = re.sub(r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b", " phone ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text, remove_stopwords=True):
    tokens = re.findall(r"[a-z0-9']+", clean_text(text))
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return tokens


def jaccard_similarity(a, b):
    a_set = set(tokenize(a))
    b_set = set(tokenize(b))
    if not a_set and not b_set:
        return 0.0
    return len(a_set & b_set) / max(1, len(a_set | b_set))


def display_records(df, columns=None, n=5, title=None):
    if title:
        print_section(title)
    if columns is None:
        columns = df.columns.tolist()
    display(df.loc[:, columns].head(n))


def top_indices(scores, top_k=5):
    scores = np.asarray(scores).ravel()
    valid = np.isfinite(scores)
    valid_idx = np.where(valid)[0]
    if len(valid_idx) == 0:
        return np.array([], dtype=int)
    ordered_valid = valid_idx[np.argsort(scores[valid_idx])[::-1]]
    return ordered_valid[:top_k]


def summarize_top_results(result_df, text_col="text", max_chars=100):
    out = result_df.copy()
    if text_col in out.columns:
        out[text_col] = out[text_col].str.slice(0, max_chars)
    return out


def plot_horizontal_bar(df, label_col, value_col, title, xlabel=None, top_n=10):
    plot_df = df.sort_values(value_col, ascending=True).tail(top_n)
    plt.figure(figsize=(8, 4.5))
    plt.barh(plot_df[label_col], plot_df[value_col])
    plt.title(title)
    if xlabel:
        plt.xlabel(xlabel)
    plt.tight_layout()
    plt.show()


def plot_2d_map(coords, labels, title, annotate_top=None):
    plt.figure(figsize=(8, 5.5))
    plt.scatter(coords[:, 0], coords[:, 1], alpha=0.65)
    if annotate_top is None:
        annotate_top = range(len(labels))
    for idx in annotate_top:
        plt.text(coords[idx, 0], coords[idx, 1], str(labels[idx]), fontsize=9)
    plt.title(title)
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.tight_layout()
    plt.show()


def build_ppmi_embeddings(token_sequences, window_size=2, min_count=2, n_components=30, random_state=SEED):
    counts = Counter()
    for seq in token_sequences:
        counts.update(seq)

    vocab = sorted([token for token, count in counts.items() if count >= min_count])
    if len(vocab) < 3:
        raise ValueError("Not enough vocabulary after min_count filtering.")

    index = {token: i for i, token in enumerate(vocab)}
    cooc = np.zeros((len(vocab), len(vocab)), dtype=float)

    for seq in token_sequences:
        seq = [token for token in seq if token in index]
        for i, target in enumerate(seq):
            left = max(0, i - window_size)
            right = min(len(seq), i + window_size + 1)
            for j in range(left, right):
                if i == j:
                    continue
                cooc[index[target], index[seq[j]]] += 1.0

    total = cooc.sum()
    row_sum = cooc.sum(axis=1, keepdims=True)
    col_sum = cooc.sum(axis=0, keepdims=True)
    expected = row_sum @ col_sum / max(total, 1e-12)

    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.log((cooc + 1e-12) / (expected + 1e-12))
    ppmi = np.maximum(pmi, 0.0)
    ppmi[cooc == 0] = 0.0

    n_components = min(n_components, len(vocab) - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)
    embeddings = normalize(svd.fit_transform(ppmi))
    return vocab, embeddings, ppmi, svd


def nearest_neighbors_by_vector(label, labels, embeddings, top_k=8):
    label_to_idx = {name: i for i, name in enumerate(labels)}
    if label not in label_to_idx:
        return pd.DataFrame({"message": [f"'{label}' was not found in the embedding vocabulary."]})
    idx = label_to_idx[label]
    scores = embeddings @ embeddings[idx]
    order = np.argsort(scores)[::-1]
    order = [i for i in order if i != idx][:top_k]
    return pd.DataFrame({
        "anchor": label,
        "neighbor": [labels[i] for i in order],
        "cosine_similarity": [scores[i] for i in order]
    })


def vocabulary_coverage(query, vectorizer):
    vocab = set(vectorizer.vocabulary_.keys())
    tokens = tokenize(query)
    covered = [tok for tok in tokens if tok in vocab]
    missing = [tok for tok in tokens if tok not in vocab]
    return {
        "query": query,
        "tokens": tokens,
        "covered_tokens": covered,
        "missing_tokens": missing,
        "coverage_rate": len(covered) / max(1, len(tokens)),
    }

## 2. Synthetic business text corpus with metadata

The corpus below mimics a customer feedback repository. Each record has text plus metadata such as channel, customer segment, product line, region, and date. The `issue` column gives us a hidden classroom label so that we can evaluate retrieval quality. In a real deployment, these labels might come from human audits, support categories, or carefully designed anchor examples.

In [ ]:
# ============================================================
# 2.1 Generate a synthetic business text corpus
# ============================================================

ISSUE_LIBRARY = {
    "refund_request": [
        "I want a refund because the item arrived damaged",
        "Can I get my money back for this order",
        "The return was approved but the refund never showed up",
        "Please return my money because the product was not as described",
        "I sent the item back and still need the refund processed",
    ],
    "delayed_delivery": [
        "My package is late and the tracking page has not updated",
        "The shipment missed the promised delivery date",
        "I paid for faster shipping but the order is still in transit",
        "The delivery window passed and nobody contacted me",
        "The carrier says pending and the package has not moved",
    ],
    "login_password": [
        "I reset my password but still cannot log in",
        "The password reset link does not work for my account",
        "I am locked out after changing my password",
        "The login page keeps rejecting the code that was sent",
        "I need help accessing my account after the reset",
    ],
    "billing_charge": [
        "There is an unexpected charge on my statement",
        "I was charged twice for the same subscription",
        "The invoice includes a fee that I do not recognize",
        "Why did the billing amount increase this month",
        "My card shows a duplicate charge from your company",
    ],
    "device_charging": [
        "The battery will not charge with the cable",
        "My wearable stops charging after a few minutes",
        "The device shows charging but the battery stays low",
        "The charging port seems loose and power disconnects",
        "My tracker will not hold a charge overnight",
    ],
    "cancellation": [
        "I want to cancel before the next renewal",
        "Please stop my subscription and turn off auto renewal",
        "I need to cancel the plan before I get billed again",
        "How do I end the membership today",
        "I tried to cancel online but the button did not work",
    ],
    "product_defect": [
        "The screen cracked during normal use",
        "The product arrived with a broken hinge",
        "The app freezes every time I open the dashboard",
        "The sensor stopped working after the update",
        "The item seems defective and needs replacement",
    ],
    "sizing_fit": [
        "The size runs small and the fit is uncomfortable",
        "The jacket is much tighter than the size chart suggested",
        "The shoes fit differently from my previous order",
        "The waistband is too loose even though I ordered my normal size",
        "The product fit is inconsistent across colors",
    ],
    "subscription_renewal": [
        "When does my annual plan renew",
        "I need the renewal date for my subscription",
        "The renewal reminder did not explain the new price",
        "Can I change plans before renewal",
        "The annual renewal notice is confusing",
    ],
    "positive_experience": [
        "The support agent solved my problem quickly",
        "The product quality is excellent and delivery was smooth",
        "I love the new dashboard and the setup was easy",
        "The replacement arrived fast and worked perfectly",
        "The service team was helpful and patient",
    ],
}

ISSUE_PRODUCT_LINES = {
    "refund_request": ["home_goods", "apparel", "subscription"],
    "delayed_delivery": ["home_goods", "apparel", "electronics"],
    "login_password": ["mobile_app", "subscription"],
    "billing_charge": ["subscription", "mobile_app"],
    "device_charging": ["wearable", "electronics"],
    "cancellation": ["subscription", "mobile_app"],
    "product_defect": ["wearable", "electronics", "mobile_app"],
    "sizing_fit": ["apparel"],
    "subscription_renewal": ["subscription"],
    "positive_experience": ["home_goods", "apparel", "mobile_app", "subscription", "wearable"],
}

CHANNELS = ["support_ticket", "chat", "review", "survey", "social"]
SEGMENTS = ["new_customer", "loyal_customer", "price_sensitive", "premium_member"]
REGIONS = ["West", "Midwest", "South", "Northeast"]
PRODUCT_NAMES = {
    "home_goods": ["HomeEase kit", "CleanNest organizer", "BrightPan"],
    "apparel": ["TrailFlex jacket", "CityFit shoes", "Everyday denim"],
    "subscription": ["Plus plan", "Family plan", "Annual membership"],
    "mobile_app": ["Shoply app", "Account Hub", "Rewards app"],
    "electronics": ["NovaCam", "ChargeDock", "MiniSpeaker"],
    "wearable": ["NovaBand", "PulseLoop", "FitRing"],
}

rng = np.random.default_rng(SEED)
records = []
start_date = pd.Timestamp("2026-01-01")
issue_names = list(ISSUE_LIBRARY.keys())
issue_probs = np.array([0.13, 0.11, 0.10, 0.11, 0.09, 0.10, 0.12, 0.08, 0.07, 0.09])
issue_probs = issue_probs / issue_probs.sum()

for i in range(280):
    issue = rng.choice(issue_names, p=issue_probs)
    product_line = rng.choice(ISSUE_PRODUCT_LINES[issue])
    base = rng.choice(ISSUE_LIBRARY[issue])
    channel = rng.choice(CHANNELS, p=[0.32, 0.22, 0.20, 0.14, 0.12])
    segment = rng.choice(SEGMENTS, p=[0.28, 0.26, 0.25, 0.21])
    region = rng.choice(REGIONS)
    product = rng.choice(PRODUCT_NAMES[product_line])
    day_offset = int(rng.integers(0, 210))
    date = start_date + pd.Timedelta(days=day_offset)

    context_bits = []
    if rng.random() < 0.72:
        context_bits.append(f"Product: {product}.")
    if rng.random() < 0.50:
        context_bits.append(f"Channel note from a {segment.replace('_', ' ')}.")
    if rng.random() < 0.25:
        context_bits.append("I already contacted support once.")
    if rng.random() < 0.16:
        context_bits.append("This is urgent because I need a response today.")

    # Newer terms appear mostly in the later period. This helps us demonstrate vocabulary drift.
    if date >= pd.Timestamp("2026-05-01") and product_line == "wearable" and rng.random() < 0.55:
        context_bits.append("The new NovaBand Loop+ wording appears in recent messages.")
    if date >= pd.Timestamp("2026-05-01") and issue in ["device_charging", "product_defect"] and rng.random() < 0.35:
        context_bits.append("Customers now describe the issue as overheating or warm on wrist.")

    text = " ".join([base] + context_bits)
    priority = "high" if ("urgent" in text.lower() or issue in ["billing_charge", "device_charging", "product_defect"]) else "normal"

    records.append({
        "doc_id": f"D{i:04d}",
        "customer_id": f"C{int(rng.integers(1000, 1125))}",
        "date": date,
        "month": str(date.to_period("M")),
        "channel": channel,
        "segment": segment,
        "region": region,
        "product_line": product_line,
        "issue": issue,
        "priority": priority,
        "text": text,
    })

corpus = pd.DataFrame(records).sort_values("date").reset_index(drop=True)
corpus["text_clean"] = corpus["text"].map(clean_text)
corpus["token_count"] = corpus["text_clean"].map(lambda x: len(tokenize(x, remove_stopwords=False)))

print(corpus.shape)
display_records(
    corpus,
    columns=["doc_id", "date", "channel", "segment", "product_line", "issue", "text"],
    n=6,
    title="Synthetic customer feedback corpus"
)

In [ ]:
# ============================================================
# 2.2 Corpus profile
# ============================================================

issue_profile = (
    corpus.groupby("issue")
    .agg(records=("doc_id", "count"), avg_tokens=("token_count", "mean"))
    .reset_index()
    .sort_values("records", ascending=False)
)

display(issue_profile)
plot_horizontal_bar(issue_profile, "issue", "records", "Synthetic corpus by hidden issue", xlabel="Number of records", top_n=10)

metadata_profile = (
    corpus.groupby(["channel", "product_line"])
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
    .head(12)
)
print_section("Channel and product-line combinations")
display(metadata_profile)

## 3. Similarity as a core operation

An embedding is useful only after we decide how to compare it. Cosine similarity emphasizes direction, dot product combines direction and magnitude, Euclidean distance measures straight-line separation, and Jaccard similarity gives a simple token-overlap baseline. The first small example uses hand-written vectors so that the geometry is visible before we move to higher-dimensional text representations.

In [ ]:
# ============================================================
# 3.1 Hand-written vectors: cosine, dot product, and Euclidean distance
# ============================================================

vectors = {
    "a: I want a refund": np.array([0.80, 0.50, 0.10]),
    "b: Please return my money": np.array([0.70, 0.60, 0.00]),
    "c: The product works great": np.array([0.10, 0.00, 0.90]),
    "d: Refund request repeated many times": np.array([2.40, 1.50, 0.30]),
}

rows = []
anchor_name = "a: I want a refund"
anchor = vectors[anchor_name]
for name, vec in vectors.items():
    dot = float(anchor @ vec)
    cosine = float((anchor @ vec) / (np.linalg.norm(anchor) * np.linalg.norm(vec)))
    euclid = float(np.linalg.norm(anchor - vec))
    rows.append({
        "comparison": f"{anchor_name} vs {name}",
        "dot_product": dot,
        "cosine_similarity": cosine,
        "euclidean_distance": euclid,
    })

similarity_demo = pd.DataFrame(rows)
display(similarity_demo)

print("After unit normalization, dot product equals cosine similarity.")
unit_vectors = {name: vec / np.linalg.norm(vec) for name, vec in vectors.items()}
normalized_dot = float(unit_vectors[anchor_name] @ unit_vectors["b: Please return my money"])
print(f"Normalized dot product for a and b: {normalized_dot:.3f}")

In [ ]:
# ============================================================
# 3.2 Jaccard overlap as an interpretable lexical baseline
# ============================================================

jaccard_examples = [
    ("I want my money back", "Can I get a refund for this order"),
    ("I want my money back", "The battery will not charge"),
    ("unexpected charge on my bill", "the device will not charge"),
]

jaccard_df = pd.DataFrame([
    {"text_a": a, "text_b": b, "jaccard_similarity": jaccard_similarity(a, b)}
    for a, b in jaccard_examples
])
display(jaccard_df)

## 4. Sparse lexical vectors versus dense document embeddings

A sparse lexical vector represents text by the words or n-grams it contains. This is transparent, but it can miss paraphrases. A dense embedding compresses text into fewer dimensions so that indirect relatedness can be captured. In this offline notebook, we use TF-IDF plus truncated SVD as a simple dense representation. In production, this encoder could be replaced by a sentence embedding model or a commercial embedding API, but the workflow logic would remain the same.

In [ ]:
# ============================================================
# 4.1 Time-respecting split and train-only encoders
# ============================================================

cutoff_date = pd.Timestamp("2026-05-01")
train_mask = corpus["date"] < cutoff_date
train_df = corpus.loc[train_mask].copy().reset_index(drop=True)
future_df = corpus.loc[~train_mask].copy().reset_index(drop=True)

print(f"Training records before {cutoff_date.date()}: {len(train_df)}")
print(f"Future records on or after {cutoff_date.date()}: {len(future_df)}")

# The vectorizer is fit only on the training period. This makes vocabulary drift visible.
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    sublinear_tf=True,
)

X_train_tfidf = normalize(tfidf_vectorizer.fit_transform(train_df["text_clean"]))
X_all_tfidf = normalize(tfidf_vectorizer.transform(corpus["text_clean"]))

n_components = min(45, X_train_tfidf.shape[1] - 1)
svd_encoder = TruncatedSVD(n_components=n_components, random_state=SEED)
X_train_dense = normalize(svd_encoder.fit_transform(X_train_tfidf))
X_all_dense = normalize(svd_encoder.transform(X_all_tfidf))

print(f"TF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_):,}")
print(f"Dense embedding dimension: {X_all_dense.shape[1]}")
print(f"Explained variance ratio from SVD: {svd_encoder.explained_variance_ratio_.sum():.3f}")

In [ ]:
# ============================================================
# 4.2 Query encoding and retrieval helpers
# ============================================================

def encode_query_tfidf(query):
    return normalize(tfidf_vectorizer.transform([clean_text(query)]))


def encode_query_dense(query):
    q_tfidf = encode_query_tfidf(query)
    return normalize(svd_encoder.transform(q_tfidf))


def retrieve_tfidf(query, top_k=5, df=None, mask=None):
    if df is None:
        df = corpus
    q = encode_query_tfidf(query)
    scores = cosine_similarity(q, X_all_tfidf).ravel()
    if mask is not None:
        scores = scores.copy()
        scores[~mask] = -np.inf
    idx = top_indices(scores, top_k=top_k)
    out = corpus.iloc[idx].copy()
    out["score"] = scores[idx]
    return out[["doc_id", "score", "issue", "product_line", "channel", "date", "text"]]


def retrieve_dense(query, top_k=5, mask=None):
    q = encode_query_dense(query)
    scores = (X_all_dense @ q.ravel())
    if mask is not None:
        scores = scores.copy()
        scores[~mask] = -np.inf
    idx = top_indices(scores, top_k=top_k)
    out = corpus.iloc[idx].copy()
    out["score"] = scores[idx]
    return out[["doc_id", "score", "issue", "product_line", "channel", "date", "text"]]

query = "I need my money back because the item was broken"
print_section("Sparse TF-IDF retrieval")
display(summarize_top_results(retrieve_tfidf(query, top_k=5)))

print_section("Dense SVD retrieval")
display(summarize_top_results(retrieve_dense(query, top_k=5)))

In [ ]:
# ============================================================
# 4.3 Coverage audit for new vocabulary
# ============================================================

coverage_queries = [
    "NovaBand Loop+ is warm on wrist and will not charge",
    "I was charged twice for the annual membership",
    "The package is delayed and tracking has not moved",
]
coverage_df = pd.DataFrame([vocabulary_coverage(q, tfidf_vectorizer) for q in coverage_queries])
display(coverage_df[["query", "covered_tokens", "missing_tokens", "coverage_rate"]])

## 5. Word embedding intuition from context and contrast

Classic word embeddings learn from usage. Words that appear in similar contexts move closer together. The code below builds a small co-occurrence matrix from our corpus, transforms it into a positive pointwise mutual information matrix, and compresses it with SVD. This is not Word2Vec, but it demonstrates the same core idea: context creates geometry.

In [ ]:
# ============================================================
# 5.1 Build simple word embeddings from local context windows
# ============================================================

token_sequences = [tokenize(text) for text in train_df["text_clean"]]
word_vocab, word_embeddings, ppmi_matrix, word_svd = build_ppmi_embeddings(
    token_sequences,
    window_size=2,
    min_count=3,
    n_components=25,
)

print(f"Word vocabulary after min_count filtering: {len(word_vocab)}")
for anchor in ["refund", "money", "charge", "charging", "password", "cancel", "delivery"]:
    print_section(f"Nearest words for anchor: {anchor}")
    display(nearest_neighbors_by_vector(anchor, word_vocab, word_embeddings, top_k=7))

In [ ]:
# ============================================================
# 5.2 Visualize a small word neighborhood
# ============================================================

anchor_terms = ["refund", "money", "return", "charge", "charging", "battery", "password", "login", "cancel", "renewal", "delivery"]
anchor_terms = [term for term in anchor_terms if term in word_vocab]
anchor_idx = [word_vocab.index(term) for term in anchor_terms]
word_coords = PCA(n_components=2, random_state=SEED).fit_transform(word_embeddings[anchor_idx])
plot_2d_map(word_coords, anchor_terms, "Small projection of selected word embeddings", annotate_top=range(len(anchor_terms)))

## 6. Sentence and document retrieval with decision-oriented evaluation

For most business workflows, the decision unit is not a word. It is a message, review, ticket, paragraph, product description, or policy passage. We now evaluate retrieval at the document level using anchor queries. The hidden `issue` label lets us calculate whether the top results are useful for a support or insight workflow.

In [ ]:
# ============================================================
# 6.1 Anchor queries for retrieval evaluation
# ============================================================

anchor_queries = pd.DataFrame([
    {"query_id": "Q01", "query": "I want my money back for a damaged item", "target_issue": "refund_request"},
    {"query_id": "Q02", "query": "tracking has not moved and delivery is late", "target_issue": "delayed_delivery"},
    {"query_id": "Q03", "query": "password reset link failed and I cannot access the account", "target_issue": "login_password"},
    {"query_id": "Q04", "query": "there is a duplicate charge on my card", "target_issue": "billing_charge"},
    {"query_id": "Q05", "query": "my wearable battery will not charge overnight", "target_issue": "device_charging"},
    {"query_id": "Q06", "query": "please cancel my plan before the renewal", "target_issue": "cancellation"},
    {"query_id": "Q07", "query": "the screen cracked and the sensor stopped working", "target_issue": "product_defect"},
    {"query_id": "Q08", "query": "the jacket runs small and does not fit", "target_issue": "sizing_fit"},
    {"query_id": "Q09", "query": "when does my annual subscription renew", "target_issue": "subscription_renewal"},
    {"query_id": "Q10", "query": "support solved the issue quickly and was helpful", "target_issue": "positive_experience"},
])

def evaluate_retrieval(query_df, retrieval_function, k=5):
    rows = []
    for _, row in query_df.iterrows():
        res = retrieval_function(row["query"], top_k=k)
        top_issues = res["issue"].tolist()
        rows.append({
            "query_id": row["query_id"],
            "target_issue": row["target_issue"],
            "top1_issue": top_issues[0] if top_issues else None,
            "top1_correct": int(len(top_issues) > 0 and top_issues[0] == row["target_issue"]),
            f"precision_at_{k}": np.mean([issue == row["target_issue"] for issue in top_issues]) if top_issues else 0.0,
            f"hit_at_{k}": int(row["target_issue"] in top_issues),
        })
    return pd.DataFrame(rows)

retrieval_eval = pd.concat([
    evaluate_retrieval(anchor_queries, retrieve_tfidf, k=5).assign(method="sparse_tfidf"),
    evaluate_retrieval(anchor_queries, retrieve_dense, k=5).assign(method="dense_svd"),
], ignore_index=True)

display(retrieval_eval)

summary_eval = (
    retrieval_eval.groupby("method")
    .agg(top1_accuracy=("top1_correct", "mean"), precision_at_5=("precision_at_5", "mean"), hit_at_5=("hit_at_5", "mean"))
    .reset_index()
)
print_section("Retrieval evaluation summary")
display(summary_eval)

In [ ]:
# ============================================================
# 6.2 Inspect one success and one ambiguity
# ============================================================

query_success = "password reset code does not work and I cannot log in"
print_section("Dense retrieval for a login query")
display(summarize_top_results(retrieve_dense(query_success, top_k=5)))

query_ambiguous = "the charge is wrong and I need help"
print_section("Dense retrieval for an ambiguous charge query")
display(summarize_top_results(retrieve_dense(query_ambiguous, top_k=8)))

## 7. Hybrid retrieval and metadata-aware refinement

Business relevance is rarely semantic similarity alone. Exact identifiers, product lines, source type, recency, and operational priority can matter. A hybrid workflow keeps the embedding search as candidate generation, then refines the ranking with lexical overlap and metadata filters.

In [ ]:
# ============================================================
# 7.1 Hybrid retrieval: dense similarity plus sparse similarity plus metadata filtering
# ============================================================

def retrieve_hybrid(query, top_k=5, product_line=None, channel=None, alpha=0.65):
    q_tfidf = encode_query_tfidf(query)
    q_dense = encode_query_dense(query)

    sparse_scores = cosine_similarity(q_tfidf, X_all_tfidf).ravel()
    dense_scores = X_all_dense @ q_dense.ravel()
    scores = alpha * dense_scores + (1 - alpha) * sparse_scores

    mask = np.ones(len(corpus), dtype=bool)
    if product_line is not None:
        mask &= corpus["product_line"].eq(product_line).to_numpy()
    if channel is not None:
        mask &= corpus["channel"].eq(channel).to_numpy()

    scores = scores.copy()
    scores[~mask] = -np.inf
    idx = top_indices(scores, top_k=top_k)
    out = corpus.iloc[idx].copy()
    out["hybrid_score"] = scores[idx]
    out["dense_score"] = dense_scores[idx]
    out["sparse_score"] = sparse_scores[idx]
    return out[["doc_id", "hybrid_score", "dense_score", "sparse_score", "issue", "product_line", "channel", "date", "text"]]

billing_query = "the charge on my card looks wrong"
device_query = "the device will not charge with the cable"

print_section("Ambiguous charge query without product filter")
display(summarize_top_results(retrieve_hybrid(billing_query, top_k=6)))

print_section("Charge query refined to subscription product line")
display(summarize_top_results(retrieve_hybrid(billing_query, top_k=6, product_line="subscription")))

print_section("Charge query refined to wearable product line")
display(summarize_top_results(retrieve_hybrid(device_query, top_k=6, product_line="wearable")))

In [ ]:
# ============================================================
# 7.2 Compare hybrid weight choices on anchor queries
# ============================================================

def make_hybrid_function(alpha):
    return lambda query, top_k=5: retrieve_hybrid(query, top_k=top_k, alpha=alpha)

weight_rows = []
for alpha in [0.00, 0.35, 0.65, 0.85, 1.00]:
    metrics = evaluate_retrieval(anchor_queries, make_hybrid_function(alpha), k=5)
    weight_rows.append({
        "dense_weight_alpha": alpha,
        "top1_accuracy": metrics["top1_correct"].mean(),
        "precision_at_5": metrics["precision_at_5"].mean(),
        "hit_at_5": metrics["hit_at_5"].mean(),
    })

weight_eval = pd.DataFrame(weight_rows)
display(weight_eval)

## 8. Matching and threshold calibration

Retrieval returns a ranked list. Matching usually requires a decision rule: are these two records similar enough to be treated as the same underlying issue? This threshold should not be selected because it looks neat. It should reflect the cost of false matches and missed matches.

In [ ]:
# ============================================================
# 8.1 Create synthetic ticket pairs for matching evaluation
# ============================================================

rng = np.random.default_rng(SEED + 10)
positive_pairs = []
negative_pairs = []

for issue, group in corpus.groupby("issue"):
    group_indices = group.index.to_numpy()
    if len(group_indices) >= 2:
        for _ in range(18):
            i, j = rng.choice(group_indices, size=2, replace=False)
            positive_pairs.append((i, j, 1))

all_indices = corpus.index.to_numpy()
while len(negative_pairs) < len(positive_pairs):
    i, j = rng.choice(all_indices, size=2, replace=False)
    if corpus.loc[i, "issue"] != corpus.loc[j, "issue"]:
        negative_pairs.append((i, j, 0))

pair_df = pd.DataFrame(positive_pairs + negative_pairs, columns=["idx_a", "idx_b", "same_issue"])
pair_df = pair_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Similarity features.
pair_df["dense_cosine"] = [float(X_all_dense[i] @ X_all_dense[j]) for i, j in zip(pair_df["idx_a"], pair_df["idx_b"])]
pair_df["sparse_cosine"] = [float(X_all_tfidf[i].multiply(X_all_tfidf[j]).sum()) for i, j in zip(pair_df["idx_a"], pair_df["idx_b"])]
pair_df["jaccard"] = [jaccard_similarity(corpus.loc[i, "text"], corpus.loc[j, "text"]) for i, j in zip(pair_df["idx_a"], pair_df["idx_b"])]
pair_df["combined_score"] = 0.60 * pair_df["dense_cosine"] + 0.30 * pair_df["sparse_cosine"] + 0.10 * pair_df["jaccard"]

pair_df["issue_a"] = pair_df["idx_a"].map(corpus["issue"])
pair_df["issue_b"] = pair_df["idx_b"].map(corpus["issue"])

display(pair_df.head(8)[["same_issue", "issue_a", "issue_b", "dense_cosine", "sparse_cosine", "jaccard", "combined_score"]])

In [ ]:
# ============================================================
# 8.2 Threshold table with explicit error costs
# ============================================================

threshold_rows = []
y_true = pair_df["same_issue"].to_numpy()
score = pair_df["combined_score"].to_numpy()

false_positive_cost = 5.0  # incorrectly merging different issues can be expensive
false_negative_cost = 1.0  # missing a duplicate is still costly, but usually less severe here

for threshold in np.round(np.linspace(0.10, 0.90, 9), 2):
    pred = (score >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    cost = false_positive_cost * fp + false_negative_cost * fn
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "false_positives": fp,
        "false_negatives": fn,
        "business_cost": cost,
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

best_row = threshold_df.sort_values("business_cost").iloc[0]
print(f"Cost-sensitive threshold suggestion: {best_row['threshold']:.2f}")

In [ ]:
# ============================================================
# 8.3 Inspect borderline pairs near the suggested threshold
# ============================================================

selected_threshold = float(best_row["threshold"])
pair_df["distance_from_threshold"] = (pair_df["combined_score"] - selected_threshold).abs()
borderline = pair_df.sort_values("distance_from_threshold").head(6).copy()

inspection_rows = []
for _, row in borderline.iterrows():
    inspection_rows.append({
        "same_issue": row["same_issue"],
        "combined_score": row["combined_score"],
        "issue_a": row["issue_a"],
        "issue_b": row["issue_b"],
        "text_a": corpus.loc[row["idx_a"], "text"][:110],
        "text_b": corpus.loc[row["idx_b"], "text"][:110],
    })

display(pd.DataFrame(inspection_rows))

## 9. Item, customer, and event embeddings beyond text

The same context logic applies beyond language. Products can learn from baskets, customers can learn from repeated behavior, and touchpoints can learn from journeys. The key question is not whether the object is text. The key question is what context should define relatedness.

In [ ]:
# ============================================================
# 9.1 Synthetic product baskets and item embeddings
# ============================================================

PRODUCT_CATALOG = pd.DataFrame([
    {"item": "whole_grain_cereal", "category": "cereal", "mission": "breakfast"},
    {"item": "honey_cereal", "category": "cereal", "mission": "breakfast"},
    {"item": "dairy_milk", "category": "milk", "mission": "breakfast"},
    {"item": "oat_milk", "category": "milk", "mission": "breakfast"},
    {"item": "banana", "category": "fruit", "mission": "breakfast"},
    {"item": "coffee_beans", "category": "coffee", "mission": "coffee"},
    {"item": "coffee_filter", "category": "coffee", "mission": "coffee"},
    {"item": "creamer", "category": "dairy", "mission": "coffee"},
    {"item": "travel_mug", "category": "accessory", "mission": "coffee"},
    {"item": "pasta", "category": "meal_base", "mission": "dinner"},
    {"item": "tomato_sauce", "category": "sauce", "mission": "dinner"},
    {"item": "parmesan", "category": "cheese", "mission": "dinner"},
    {"item": "olive_oil", "category": "oil", "mission": "dinner"},
    {"item": "laundry_detergent", "category": "cleaning", "mission": "laundry"},
    {"item": "fabric_softener", "category": "cleaning", "mission": "laundry"},
    {"item": "dryer_sheets", "category": "cleaning", "mission": "laundry"},
    {"item": "stain_remover", "category": "cleaning", "mission": "laundry"},
    {"item": "chips", "category": "snack", "mission": "snack"},
    {"item": "salsa", "category": "dip", "mission": "snack"},
    {"item": "sparkling_water", "category": "beverage", "mission": "snack"},
])

mission_to_items = PRODUCT_CATALOG.groupby("mission")["item"].apply(list).to_dict()
category_to_items = PRODUCT_CATALOG.groupby("category")["item"].apply(list).to_dict()

rng = np.random.default_rng(SEED + 20)
basket_records = []
for basket_id in range(500):
    customer_id = f"B{int(rng.integers(1000, 1120))}"
    mission = rng.choice(list(mission_to_items.keys()), p=[0.26, 0.18, 0.20, 0.18, 0.18])
    items = []
    # Draw from the mission, then occasionally add a cross-mission item.
    mission_items = mission_to_items[mission]
    n_items = int(rng.integers(2, min(5, len(mission_items)) + 1))
    items.extend(rng.choice(mission_items, size=n_items, replace=False).tolist())
    if rng.random() < 0.18:
        other_mission = rng.choice([m for m in mission_to_items if m != mission])
        items.append(rng.choice(mission_to_items[other_mission]))
    basket_records.append({
        "basket_id": f"K{basket_id:04d}",
        "customer_id": customer_id,
        "mission": mission,
        "items": sorted(set(items)),
    })

baskets = pd.DataFrame(basket_records)
display(baskets.head(8))

item_sequences = baskets["items"].tolist()
item_vocab, item_embeddings, item_ppmi, item_svd = build_ppmi_embeddings(
    item_sequences,
    window_size=10,
    min_count=2,
    n_components=8,
)

print_section("Nearest item neighbors for whole_grain_cereal")
item_neighbors = nearest_neighbors_by_vector("whole_grain_cereal", item_vocab, item_embeddings, top_k=8)
item_neighbors = item_neighbors.merge(PRODUCT_CATALOG, left_on="neighbor", right_on="item", how="left").drop(columns=["item"])
display(item_neighbors)

In [ ]:
# ============================================================
# 9.2 Visualize an item embedding neighborhood
# ============================================================

item_coords = PCA(n_components=2, random_state=SEED).fit_transform(item_embeddings)
# Annotate all items because the catalog is small.
plot_2d_map(item_coords, item_vocab, "Projection of product item embeddings", annotate_top=range(len(item_vocab)))

In [ ]:
# ============================================================
# 9.3 Customer embeddings from average item behavior
# ============================================================

item_to_idx = {item: i for i, item in enumerate(item_vocab)}
customer_vectors = {}
customer_missions = defaultdict(Counter)
for _, row in baskets.iterrows():
    vectors = [item_embeddings[item_to_idx[item]] for item in row["items"] if item in item_to_idx]
    if vectors:
        customer_vectors.setdefault(row["customer_id"], []).extend(vectors)
        customer_missions[row["customer_id"]][row["mission"]] += 1

customer_ids = sorted(customer_vectors.keys())
customer_embeddings = np.vstack([np.mean(customer_vectors[c], axis=0) for c in customer_ids])
customer_embeddings = normalize(customer_embeddings)

focal_customer = customer_ids[0]
customer_neighbors = nearest_neighbors_by_vector(focal_customer, customer_ids, customer_embeddings, top_k=6)
customer_neighbors["focal_top_missions"] = [dict(customer_missions[focal_customer].most_common(2))] * len(customer_neighbors)
customer_neighbors["neighbor_top_missions"] = customer_neighbors["neighbor"].map(lambda c: dict(customer_missions[c].most_common(2)))

display(customer_neighbors)

In [ ]:
# ============================================================
# 9.4 Event embeddings from journey contexts
# ============================================================

JOURNEY_TEMPLATES = [
    ["ad_impression", "site_visit", "product_view", "add_to_cart", "checkout", "purchase"],
    ["email_open", "site_visit", "product_view", "comparison", "add_to_cart", "checkout"],
    ["site_visit", "search", "product_view", "support_chat", "purchase"],
    ["purchase", "delivery_update", "support_chat", "return_request", "refund_processed"],
    ["ad_impression", "site_visit", "search", "product_view", "exit"],
    ["email_open", "renewal_page", "billing_page", "support_chat", "cancel_request"],
]

rng = np.random.default_rng(SEED + 30)
journeys = []
for session_id in range(320):
    template = list(JOURNEY_TEMPLATES[int(rng.integers(0, len(JOURNEY_TEMPLATES)))])
    if rng.random() < 0.20:
        template.insert(min(len(template), int(rng.integers(1, len(template)))), "promo_view")
    if rng.random() < 0.12:
        template.append("review_written")
    journeys.append(template)

event_vocab, event_embeddings, event_ppmi, event_svd = build_ppmi_embeddings(
    journeys,
    window_size=2,
    min_count=2,
    n_components=6,
)

for event in ["support_chat", "purchase", "cancel_request", "return_request"]:
    print_section(f"Nearest event contexts for {event}")
    display(nearest_neighbors_by_vector(event, event_vocab, event_embeddings, top_k=5))

## 10. Operational checkpoints for embedding systems

Embedding workflows should be monitored like measurement systems. The basic questions are practical: Are anchor queries still retrieving the right material? Does the vocabulary cover new business language? Are results stable after model or index updates? Are personal or sensitive details being encoded unnecessarily? Can a manager understand what the similarity score means and what it does not mean?

In [ ]:
# ============================================================
# 10.1 Anchor acceptance tests
# ============================================================

anchor_acceptance = retrieval_eval.query("method == 'dense_svd'").copy()
anchor_acceptance["status"] = np.where(
    (anchor_acceptance["hit_at_5"] == 1) & (anchor_acceptance["precision_at_5"] >= 0.40),
    "pass",
    "review",
)

display(anchor_acceptance[["query_id", "target_issue", "top1_issue", "precision_at_5", "hit_at_5", "status"]])

print_section("Acceptance status counts")
display(anchor_acceptance["status"].value_counts().rename_axis("status").reset_index(name="queries"))

In [ ]:
# ============================================================
# 10.2 Vocabulary and drift signals
# ============================================================

late_queries = pd.DataFrame([
    vocabulary_coverage("NovaBand Loop+ is overheating and warm on wrist", tfidf_vectorizer),
    vocabulary_coverage("duplicate annual membership charge on my card", tfidf_vectorizer),
    vocabulary_coverage("carrier tracking has not moved and package is late", tfidf_vectorizer),
])

display(late_queries[["query", "covered_tokens", "missing_tokens", "coverage_rate"]])

monthly_issue_terms = (
    corpus.assign(has_novaband=corpus["text_clean"].str.contains("novaband"), has_overheat=corpus["text_clean"].str.contains("overheat|warm"))
    .groupby("month")
    .agg(records=("doc_id", "count"), novaband_mentions=("has_novaband", "sum"), overheating_terms=("has_overheat", "sum"))
    .reset_index()
)
display(monthly_issue_terms)

plt.figure(figsize=(8, 4.5))
plt.plot(monthly_issue_terms["month"], monthly_issue_terms["novaband_mentions"], marker="o", label="NovaBand mentions")
plt.plot(monthly_issue_terms["month"], monthly_issue_terms["overheating_terms"], marker="o", label="Overheating or warm terms")
plt.title("Simple vocabulary drift signals by month")
plt.xlabel("Month")
plt.ylabel("Mentions")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 10.3 Segment and channel coverage audit
# ============================================================

coverage_audit = (
    corpus.groupby(["segment", "channel"])
    .agg(records=("doc_id", "count"), high_priority_rate=("priority", lambda x: np.mean(x == "high")))
    .reset_index()
    .sort_values("records", ascending=True)
)

display(coverage_audit.head(12))

segment_summary = (
    corpus.groupby("segment")
    .agg(records=("doc_id", "count"), issues=("issue", "nunique"), high_priority_rate=("priority", lambda x: np.mean(x == "high")))
    .reset_index()
    .sort_values("records")
)
display(segment_summary)

In [ ]:
# ============================================================
# 10.4 Privacy-aware similarity checklist
# ============================================================

privacy_patterns = {
    "email_like": r"[\w\.-]+@[\w\.-]+",
    "phone_like": r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b",
    "account_id_like": r"\b(account|case|ticket)\s*#?\s*\d{4,}\b",
}

privacy_rows = []
for name, pattern in privacy_patterns.items():
    privacy_rows.append({
        "risk_signal": name,
        "records_flagged": int(corpus["text"].str.contains(pattern, case=False, regex=True).sum()),
        "why_it_matters": "Identifiers can create memorization, reidentification, or inappropriate matching risk.",
        "typical_response": "Replace with placeholders before embedding and store raw text access separately.",
    })

privacy_audit = pd.DataFrame(privacy_rows)
display(privacy_audit)

In [ ]:
# ============================================================
# 10.5 Embedding system card
# ============================================================

embedding_system_card = pd.DataFrame([
    {"field": "Business decision", "entry": "Retrieve and match customer feedback for support triage and insight discovery."},
    {"field": "Unit of representation", "entry": "One customer feedback record. Long records would be chunked before embedding."},
    {"field": "Encoder used in this notebook", "entry": "Train-period TF-IDF followed by truncated SVD and row normalization."},
    {"field": "Similarity metric", "entry": "Cosine similarity through unit-normalized vectors."},
    {"field": "Training window", "entry": f"Records dated before {cutoff_date.date()}."},
    {"field": "Known limitation", "entry": "New terms after the cutoff may be missing from the vocabulary."},
    {"field": "Operational guardrail", "entry": "Run anchor tests, vocabulary coverage checks, threshold audits, and privacy screening before deployment."},
    {"field": "Human review trigger", "entry": "Borderline matches and high-cost false positives require review."},
])

display(embedding_system_card)

## Decision guide

Use sparse lexical vectors when the task depends on exact words, identifiers, or transparent baselines. Use dense sentence or document embeddings when the task depends on intent, paraphrase, or semantic relatedness. Use hybrid retrieval when business relevance includes both meaning and exact constraints such as product line, source type, or geography. Use item embeddings when co-occurrence contexts such as baskets, sessions, or journeys define relatedness. Treat every embedding workflow as a measurement system that needs anchor tests, threshold calibration, drift monitoring, and documentation.

In [ ]:
# ============================================================
# Compact decision guide as a classroom reference table
# ============================================================

decision_guide = pd.DataFrame([
    {"workflow": "Semantic search", "representation": "sentence or document embeddings", "primary risk": "top results feel related but do not answer the query", "first diagnostic": "anchor queries with precision at k"},
    {"workflow": "Ticket deduplication", "representation": "hybrid pair similarity", "primary risk": "false merges across different issues", "first diagnostic": "threshold table with false-positive cost"},
    {"workflow": "Brand or issue vocabulary expansion", "representation": "word embeddings", "primary risk": "ambiguous words blend meanings", "first diagnostic": "nearest-neighbor inspection by anchor term"},
    {"workflow": "Basket neighborhood analysis", "representation": "item embeddings", "primary risk": "promotions or assortment create artificial proximity", "first diagnostic": "compare neighbors with merchandising context"},
    {"workflow": "Customer similarity", "representation": "customer embeddings", "primary risk": "behavioral proximity is mistaken for identity or cause", "first diagnostic": "segment coverage and governance review"},
])

display(decision_guide)

## Exercises

**Exercise 1: Anchor testing.** Add three new anchor queries that reflect a real business decision, such as support triage, product defect monitoring, or cancellation prevention. Evaluate sparse, dense, and hybrid retrieval. Which method would you trust, and what evidence supports that choice?

**Exercise 2: Threshold calibration.** Change the false-positive and false-negative costs in the matching section. How does the recommended threshold change? Explain the managerial reason for your cost assumptions.

**Exercise 3: Unit of representation.** Split longer feedback records into shorter chunks before embedding. Does retrieval improve for mixed-topic records? Report one example where chunking helps and one where it hurts.

**Exercise 4: Product embeddings.** Add a new product mission such as pet care, school lunch, or home office. Generate baskets, rebuild item embeddings, and inspect the nearest neighbors of one focal item.

**Exercise 5: Drift monitoring.** Add a new term that appears only in the future period. Run the vocabulary coverage audit and propose a model refresh policy.

## Closing note

Embedding systems are powerful because they turn many business problems into comparison problems. Their value, however, depends on disciplined measurement design. A similarity score is not a fact about the world. It is the output of a representation, a metric, a corpus, and a workflow. The analyst's job is to make those choices explicit, evaluate them against the decision, and communicate the limits of the resulting neighborhood.